# Melbourne Rainfall – Cumulative to Per-Interval Conversion
This notebook converts the `Period_Rainfall` column (cumulative since 9am) into a per-interval rainfall value representing mm fallen each ~10 minute observation window.

## 1. Load the data

In [44]:
import pandas as pd

df = pd.read_csv('cleaned_Melbourne_no_missingvalues-1.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

print('Shape:', df.shape)
df.head()

Shape: (570578, 9)


,Timestamp,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Wind Direction (x),Wind Direction (y),Period_Rainfall
0,2011-01-01 00:04:00,24.8,14.0,51.0,11.0,1007.4,-0.707107,0.707107,0.0
1,2011-01-01 00:14:00,24.8,13.3,48.0,11.0,1007.5,-0.707107,0.707107,0.0
2,2011-01-01 00:24:00,24.9,13.3,48.0,11.0,1007.5,-0.707107,0.707107,0.0
3,2011-01-01 00:34:00,24.7,13.4,49.0,11.0,1007.4,-0.707107,0.707107,0.0
4,2011-01-01 00:44:00,24.1,13.3,51.0,9.0,1007.3,-0.382683,0.923880,0.0


## 2. Inspect the original rainfall column

In [21]:
print('Rainfall stats:')
print(df['Period_Rainfall'].describe())
print()
print('Non-zero rows:', (df['Period_Rainfall'] > 0).sum())

Rainfall stats:
count    570578.000000
mean          0.014673
std           0.152765
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          25.800000
Name: Period_Rainfall, dtype: float64

Non-zero rows: 16469


## 3. Convert cumulative-since-9am to per-interval rainfall

**Logic:**
- The gauge accumulates from **9am each day** and resets the next 9am
- Group rows into 9am-to-9am periods
- `diff()` within each period gives the increase since the previous reading
- `clip(lower=0)` discards negative diffs caused by mid-period gauge overflows/resets
- `fillna` keeps the first reading of each period as-is

In [25]:
# Shift timestamps back 9 hours so date boundaries align with 9am
df['rain_period'] = (df['Timestamp'] - pd.Timedelta(hours=9)).dt.normalize()

# Compute per-interval rainfall within each 9am period
df['Interval_Rainfall'] = (
    df.groupby('rain_period')['Period_Rainfall']
    .transform(lambda g: g.diff().clip(lower=0).fillna(g.iloc[0]))
)

print('Negative values (should be 0):', (df['Interval_Rainfall'] < 0).sum())
print()
print('Interval rainfall stats:')
df['Interval_Rainfall'].describe()

Negative values (should be 0): 0

Interval rainfall stats:


count    570578.000000
mean          0.008120
std           0.114455
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          25.800000
Name: Interval_Rainfall, dtype: float64

## 4. Verify the conversion on a known heavy rain event

In [27]:
# Dec 14 2018 – gauge overflow event (21.4mm then reset to 2.0)
mask = (df['Timestamp'] >= '2018-12-14 16:30') & (df['Timestamp'] <= '2018-12-14 18:30')
df[mask][['Timestamp', 'Period_Rainfall', 'Interval_Rainfall']]

,Timestamp,Period_Rainfall,Interval_Rainfall
364693,2018-12-14 16:30:00,0.0,0.0
364694,2018-12-14 16:40:00,0.0,0.0
364695,2018-12-14 16:50:00,0.0,0.0
364696,2018-12-14 17:00:00,0.0,0.0
364697,2018-12-14 17:10:00,0.2,0.2
364698,2018-12-14 17:20:00,13.6,13.4
364699,2018-12-14 17:30:00,21.4,7.8
364700,2018-12-14 17:40:00,2.0,0.0
364701,2018-12-14 18:00:00,0.2,0.0
364702,2018-12-14 18:10:00,0.0,0.0


## 5. Tidy up columns and save

In [29]:
df = df.drop(columns=['rain_period', 'Period_Rainfall'])
df = df.rename(columns={'Interval_Rainfall': 'Rainfall (mm)'})

cols = [
    'Timestamp',
    'Air Temp (degrees C)',
    'Dew Pt Temp (degrees C)',
    'Humidity (%)',
    'Wind Speed (km/h)',
    'MSLP (hPa)',
    'Wind Direction (x)',
    'Wind Direction (y)',
    'Rainfall (mm)',
]
df = df[cols]

output_path = 'Melbourne_interval_rainfall.csv'
df.to_csv(output_path, index=False)

print(f'Saved to {output_path}')
print(f'Final shape: {df.shape}')
df.head(10)

Saved to Melbourne_interval_rainfall.csv
Final shape: (570578, 9)


,Timestamp,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Wind Direction (x),Wind Direction (y),Rainfall (mm)
0,2011-01-01 00:04:00,24.8,14.0,51.0,11.0,1007.4,-0.707107,0.707107,0.0
1,2011-01-01 00:14:00,24.8,13.3,48.0,11.0,1007.5,-0.707107,0.707107,0.0
2,2011-01-01 00:24:00,24.9,13.3,48.0,11.0,1007.5,-0.707107,0.707107,0.0
3,2011-01-01 00:34:00,24.7,13.4,49.0,11.0,1007.4,-0.707107,0.707107,0.0
4,2011-01-01 00:44:00,24.1,13.3,51.0,9.0,1007.3,-0.382683,0.923880,0.0
5,2011-01-01 00:54:00,24.3,13.3,50.0,9.0,1007.3,0.707107,0.707107,0.0
6,2011-01-01 01:04:00,24.1,13.3,51.0,7.0,1007.3,0.923880,0.382683,0.0
7,2011-01-01 01:14:00,23.5,13.6,53.0,7.0,1007.3,0.382683,0.923880,0.0
8,2011-01-01 01:24:00,23.3,13.5,54.0,9.0,1007.3,0.382683,0.923880,0.0
9,2011-01-01 01:34:00,23.6,13.5,53.0,6.0,1007.3,0.382683,0.923880,0.0


Converting into 1hour intervals



In [33]:
df = df.set_index('Timestamp')

hourly = df.resample('1h').agg({
    'Air Temp (degrees C)'     : 'mean',
    'Dew Pt Temp (degrees C)'  : 'mean',
    'Humidity (%)'             : 'mean',
    'Wind Speed (km/h)'        : 'mean',
    'MSLP (hPa)'               : 'mean',
    'Wind Direction (x)'       : 'mean',
    'Wind Direction (y)'       : 'mean',
    'Rainfall (mm)'            : 'sum',
})

# Round to 2 decimal places for cleanliness
hourly = hourly.round(2)
hourly = hourly.reset_index()

In [37]:
hourly

,Timestamp,Air Temp (degrees C),Dew Pt Temp (degrees C),Humidity (%),Wind Speed (km/h),MSLP (hPa),Wind Direction (x),Wind Direction (y),Rainfall (mm)
0,2011-01-01 00:00:00,24.60,13.43,49.50,10.33,1007.40,-0.42,0.74,0.0
1,2011-01-01 01:00:00,23.58,13.46,52.80,5.80,1007.28,0.41,0.63,0.0
2,2011-01-01 02:00:00,23.12,13.58,54.60,0.00,1007.26,0.00,0.00,0.0
3,2011-01-01 03:00:00,20.87,14.63,67.33,5.33,1007.57,-0.47,-0.47,0.0
4,2011-01-01 04:00:00,18.56,14.74,78.00,13.40,1008.08,-0.94,0.31,0.0
...,...,...,...,...,...,...,...,...,...
124428,2025-03-12 12:00:00,29.87,14.40,38.33,36.33,1017.87,0.95,0.26,0.0
124429,2025-03-12 13:00:00,32.10,15.13,35.67,33.00,1016.83,0.97,0.13,0.0
124430,2025-03-12 14:00:00,32.43,14.70,34.33,33.33,1015.93,0.95,0.26,0.0
124431,2025-03-12 15:00:00,33.23,14.60,32.00,22.00,1014.73,0.78,-0.60,0.0


In [41]:
# hourly.to_csv('hourly_data.csv')